In [ ]:
import os

### Basic QA / Model & Agent

In [ ]:
from langchain.chat_models import init_chat_model
model = init_chat_model('gpt-4o-mini', temperature=0)

# Or 

from langchain_openai import ChatOpenAI

# model = ChatOpenAI(model="gpt-5", temperature=0.1, max_tokens=1000, timeout=30)

In [ ]:
response = model.invoke("Why do parrots talk?")
print(response.content)   

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def search(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"

@tool
def get_weather(location: str) -> str:
    """Get weather information for a location."""
    return f"Weather in {location}: Sunny, 72°F"

agent = create_agent(model, tools=[search, get_weather])

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in San Francisco?"}]}
)

### Basic of Tools and Tool Calling in Model/Agent

In [ ]:

from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

results = search_tool.invoke('top news in india today')

print(results)
print(search_tool.name)
print(search_tool.description)
print(search_tool.args)

In [ ]:
# Method 1
from langchain_core.tools import tool

@tool
def multiply(a: int, b:int) -> int:
    """Multiply two numbers"""
    return a*b

result = multiply.invoke({"a":3, "b":5})
print(result)
print(multiply.name)
print(multiply.description)
print(multiply.args)
print(multiply.args_schema.model_json_schema())

In [ ]:
# Method 2 Base
from langchain_core.tools import tool  # New import location
from pydantic import BaseModel, Field

class MultiplyInput(BaseModel):
    a: int = Field(description="The first number to multiply")  # Fixed description
    b: int = Field(description="The second number to multiply")

@tool(args_schema=MultiplyInput)  # Single decorator does it all
def multiply_func(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

# Usage unchanged
result = multiply_func.invoke({"a": 3, "b": 3})
print(result)  # 9
print(multiply_func.name)  # multiply_func
print(multiply_func.description)  # Multiply two numbers.
print(multiply_func.args)  # {'a': ..., 'b': ...}

In [ ]:
# Toolkit
from langchain_core.tools import tool

# Custom tools
@tool
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b

class MathToolkit:
    def get_tools(self):
        return [add, multiply]

toolkit = MathToolkit()
tools = toolkit.get_tools()

for tool in tools:
    print(tool.name, "=>", tool.description)

#### Tool Binding & Calling

In [ ]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    return f"It's sunny in {location}."


model_with_tools = model.bind_tools([get_weather])

response = model_with_tools.invoke("What's the weather like in Boston?")
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

In [ ]:
@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b

model_with_tools = model.bind_tools([multiply])

In [ ]:
# result = model_with_tools.invoke('Hi how are you')
result = model_with_tools.invoke('Can you multiply 3 with 1000?')
# print(result.content)
result.tool_calls

In [ ]:
multiply.invoke(result.tool_calls[0]['args'])
# multiply.invoke(result.tool_calls[0])

In [ ]:
@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b

llm_with_tools = llm.bind_tools([multiply])

llm_with_tools.invoke('Hi how are you')

query = HumanMessage('can you multiply 3 with 1000')
messages = [query]
result = llm_with_tools.invoke(messages)
messages.append(result)

tool_result = multiply.invoke(result.tool_calls[0])
messages.append(tool_result)

llm_with_tools.invoke(messages).content

In [ ]:
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

@tool
def search(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"

@tool
def get_weather(location: str) -> str:
    """Get weather information for a location."""
    return f"Weather in {location}: Sunny, 72°F"

agent = create_agent(model, tools=[search, get_weather])

result = agent.invoke({
    "messages": [HumanMessage(content="What's the weather in SF?")]
})
print(result["messages"][-1].content)

### Messages & Prompts

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessage, SystemMessage

model = init_chat_model("gpt-5-nano")

system_msg = SystemMessage("You are a helpful assistant.")
human_msg = HumanMessage("Hello, how are you?")

# Use with chat models
messages = [system_msg, human_msg]
response = model.invoke(messages)  # Returns AIMessage
print(response.content)

In [ ]:
messages = [
    SystemMessage("You are a poetry expert"),
    HumanMessage("Write a haiku about spring"),
    AIMessage("Cherry blossoms bloom...")
]
response = model.invoke(messages)
print(response.content)

In [ ]:
messages = [
    {"role": "system", "content": "You are a poetry expert"},
    {"role": "user", "content": "Write a haiku about spring"},
    {"role": "assistant", "content": "Cherry blossoms bloom..."}
]
response = model.invoke(messages)
print(response.content)

In [ ]:
# Metadata attributes

human_msg = HumanMessage(
    content="Hello!",
    name="alice",  # Optional: identify different users
    id="msg_123",  # Optional: unique identifier for tracing
)

In [ ]:
# Tool Calls

from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5-nano")

def get_weather(location: str) -> str:
    """Get the weather at a location."""
    ...

model_with_tools = model.bind_tools([get_weather])
response = model_with_tools.invoke("What's the weather in Paris?")

for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")
    print(f"ID: {tool_call['id']}")

In [ ]:
# Tool Messages

from langchain.messages import ToolMessage

# After a model makes a tool call
# (Here, we demonstrate manually creating the messages for brevity)
ai_message = AIMessage(
    content=[],
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "San Francisco"},
        "id": "call_123"
    }]
)

# Execute tool and create result message
weather_result = "Sunny, 72°F"
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id="call_123"  # Must match the call ID
)

# Continue conversation
messages = [
    HumanMessage("What's the weather in San Francisco?"),
    ai_message,  # Model's tool call
    tool_message,  # Tool execution result
]
response = model.invoke(messages)

In [ ]:
# Content attribute , content type

# String content
human_message = HumanMessage("Hello, how are you?")

# Provider-native format (e.g., OpenAI)
human_message = HumanMessage(content=[
    {"type": "text", "text": "Hello, how are you?"},
    {"type": "image_url", "image_url": {"url": "https://example.com/image.jpg"}}
])

# List of standard content blocks
human_message = HumanMessage(content_blocks=[
    {"type": "text", "text": "Hello, how are you?"},
    {"type": "image", "url": "https://example.com/image.jpg"},
])

#### Prompts

In [ ]:
from langchain_core.prompts import PromptTemplate

# 1. Create template with variables
template = PromptTemplate.from_template(
    "Tell me a {adjective} joke about {topic}. "
    "Answer in {language}."
)

# 2. Format with values
prompt = template.format(
    adjective="funny",
    topic="programming",
    language="Spanish"
)
print(prompt)
# Output: Tell me a funny joke about programming. Answer in Spanish.

# 3. Chain with LLM
# model = ChatOpenAI(model="gpt-4o-mini")
chain = template | model

# 4. Invoke with input dict
result = chain.invoke({
    "adjective": "silly",
    "topic": "chickens",
    "language": "English"
})
print(result.content)
# Output: Why did the chicken cross the road? To get to the silly side! 🐔😂

In [ ]:
from langchain_core.prompts import PromptTemplate,load_prompt
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini")

# Your exact code ✅
template2 = PromptTemplate(
    template='Greet this person in 5 languages. The name of the person is {name}',
    input_variables=['name']
)

prompt = template2.invoke({'name': 'nitish'})
result = model.invoke(prompt)
print(result.content)

In [ ]:
template = PromptTemplate.from_template('Greet {name} in 5 languages')
chain = template | model
result = chain.invoke({'name': 'nitish'})

# template.save('template.json')
# prompt2 = load_prompt('template.json')

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are a comedian."),
    ("human", "Tell me a {adjective} joke about {topic}.")
])

chain = chat_template | model
result = chain.invoke({"adjective": "bad", "topic": "AI"})

### Structured Output

#### with_structured_output

In [ ]:
from pydantic import BaseModel, Field
from typing import Type, TypedDict, Annotated, Optional, Literal

# - typeddict (define, no validation, kind of hint, annotated to give description)
class Review(TypedDict):

    key_themes: Annotated[list[str], "Write down all the key themes discussed in the review in a list"]
    summary: Annotated[str, "A brief summary of the review"]
    sentiment: Annotated[Literal["pos", "neg"], "Return sentiment of the review either negative, positive or neutral"]
    pros: Annotated[Optional[list[str]], "Write down all the pros inside a list"]
    cons: Annotated[Optional[list[str]], "Write down all the cons inside a list"]
    name: Annotated[Optional[str], "Write the name of the reviewer"]

In [ ]:
# - pydantic (basemodel, validation , field for description and other control)
class Review(BaseModel):

    key_themes: list[str] = Field(description="Write down all the key themes discussed in the review in a list")
    summary: str = Field(description="A brief summary of the review")
    sentiment: Literal["pos", "neg"] = Field(description="Return sentiment of the review either negative, positive or neutral")
    pros: Optional[list[str]] = Field(default=None, description="Write down all the pros inside a list")
    cons: Optional[list[str]] = Field(default=None, description="Write down all the cons inside a list")
    name: Optional[str] = Field(default=None, description="Write the name of the reviewer")

In [ ]:
# - json_schema (across language, data validation and structure schema)
json_schema = {
  "title": "Review",
  "type": "object",
  "properties": {
    "key_themes": {
      "type": "array",
      "items": {
        "type": "string"
      },
      "description": "Write down all the key themes discussed in the review in a list"
    },
    "summary": {
      "type": "string",
      "description": "A brief summary of the review"
    },
    "sentiment": {
      "type": "string",
      "enum": ["pos", "neg"],
      "description": "Return sentiment of the review either negative, positive or neutral"
    },
    "pros": {
      "type": ["array", "null"],
      "items": {
        "type": "string"
      },
      "description": "Write down all the pros inside a list"
    },
    "cons": {
      "type": ["array", "null"],
      "items": {
        "type": "string"
      },
      "description": "Write down all the cons inside a list"
    },
    "name": {
      "type": ["string", "null"],
      "description": "Write the name of the reviewer"
    }
  },
  "required": ["key_themes", "summary", "sentiment"]
}

In [ ]:
structured_model = model.with_structured_output(json_schema)
# structured_model = model.with_structured_output(Review)

In [ ]:
result = structured_model.invoke("""I recently upgraded to the Samsung Galaxy S24 Ultra, and I must say, it’s an absolute powerhouse! The Snapdragon 8 Gen 3 processor makes everything lightning fast—whether I’m gaming, multitasking, or editing photos. The 5000mAh battery easily lasts a full day even with heavy use, and the 45W fast charging is a lifesaver.

The S-Pen integration is a great touch for note-taking and quick sketches, though I don't use it often. What really blew me away is the 200MP camera—the night mode is stunning, capturing crisp, vibrant images even in low light. Zooming up to 100x actually works well for distant objects, but anything beyond 30x loses quality.

However, the weight and size make it a bit uncomfortable for one-handed use. Also, Samsung’s One UI still comes with bloatware—why do I need five different Samsung apps for things Google already provides? The $1,300 price tag is also a hard pill to swallow.

Pros:
Insanely powerful processor (great for gaming and productivity)
Stunning 200MP camera with incredible zoom capabilities
Long battery life with fast charging
S-Pen support is unique and useful
                                 
Review by Nitish Singh
""")

print(result['name'])

#### OutputParser

In [ ]:
from langchain_core.output_parsers import (
    StrOutputParser,
    JsonOutputParser,
    PydanticOutputParser
)

In [ ]:
# StrOutputParser
from langchain_core.prompts import PromptTemplate

template1 = PromptTemplate(
    template='Write a detailed report on {topic}',
    input_variables=['topic']
)

# 2nd prompt -> summary
template2 = PromptTemplate(
    template='Write a 5 line summary on the following text. /n {text}',
    input_variables=['text']
)

parser = StrOutputParser()

chain = template1 | model | parser | template2 | model | parser

result = chain.invoke({'topic':'black hole'})

print(result)

In [ ]:
# JsonOutputParser

parser = JsonOutputParser()

template = PromptTemplate(
    template='Give me 5 facts about {topic} \n {format_instruction}',
    input_variables=['topic'],
    partial_variables={'format_instruction': parser.get_format_instructions()}
)

chain = template | model | parser

result = chain.invoke({'topic':'black hole'})

print(result)

In [ ]:
# PydanticOutputParser

class Person(BaseModel):

    name: str = Field(description='Name of the person')
    age: int = Field(gt=18, description='Age of the person')
    city: str = Field(description='Name of the city the person belongs to')

parser = PydanticOutputParser(pydantic_object=Person)

template = PromptTemplate(
    template='Generate the name, age and city of a fictional {place} person \n {format_instruction}',
    input_variables=['place'],
    partial_variables={'format_instruction':parser.get_format_instructions()}
)

chain = template | model | parser

final_result = chain.invoke({'place':'sri lankan'})

print(final_result)

### Memory

### Vector store & RAG

In [ ]:
# Document Loader
# Text splitter
# Vector Store
# Retrievers
# Retrieval Augmented Generation (RAG)

#### Step 1: Your source documents

In [ ]:

from langchain_core.documents import Document

documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

In [ ]:
"""
# Document Loaders Template - LangChain Basics
Load, process, and summarize documents using various loaders.

Most common loaders for text, PDF, web, etc.
"""

from langchain_community.document_loaders import (
    TextLoader,                    # Plain text files (.txt)
    PyPDFLoader,                   # PDF documents
    UnstructuredPDFLoader,         # Complex PDFs (tables/images)
    WebBaseLoader,                 # Web pages (URLs)
    UnstructuredHTMLLoader,        # HTML files
    CSVLoader,                     # CSV data
    JSONLoader,                    # JSON files
    UnstructuredWordDocumentLoader, # Word docs (.docx)
    # Add more: UnstructuredMarkdownLoader, NotionDirectoryLoader, etc.
)

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser


# Your summarization chain
prompt = PromptTemplate(
    template='Write a concise summary for the following document:\n\n{doc}',
    input_variables=['doc']
)
parser = StrOutputParser()
chain = prompt | model | parser

# Example: Load and process
loader = TextLoader('random.txt', encoding='utf-8')
docs = loader.load()


print(f"Loaded {len(docs)} documents")
print(f"Type: {type(docs)}")
print("\nFirst doc content:")
print(docs[0].page_content[:200] + "...")
print("\nMetadata:", docs[0].metadata)

# Summarize
summary = chain.invoke({'doc': docs[0].page_content})
print(f"\nSummary:\n{summary}")

# Usage notes:
# - docs is List[Document] with .page_content and .metadata
# - Try PyPDFLoader("file.pdf") for PDFs
# - WebBaseLoader("https://example.com") for web
# - Split docs: from langchain_text_splitters import RecursiveCharacterTextSplitter

# docs = loader.lazy_load()
# for document in docs:
    # print(document.metadata)

In [ ]:
"""
# Text Splitters Template - LangChain Basics
Split documents into chunks for RAG/embeddings.

Key splitters: character, recursive, semantic.
"""

from langchain_text_splitters import (           # New unified package
    CharacterTextSplitter,           # Fixed-size chunks
    RecursiveCharacterTextSplitter,  # Smart paragraph-aware (RECOMMENDED)
    MarkdownHeaderTextSplitter,      # Markdown by headers
    PythonCodeTextSplitter,          # Code functions/classes
    TokenTextSplitter,               # Token-accurate (tiktoken)
)

from langchain_experimental.text_splitter import SemanticChunker  # Semantic splits
from langchain_openai import OpenAIEmbeddings  # For semantic

from langchain_community.document_loaders import (
    PyPDFLoader,       # PDFs
    TextLoader,        # Text files
    WebBaseLoader,     # Web pages
)

# Load document
# loader = PyPDFLoader('dl-curriculum.pdf')  # Or TextLoader/WebBaseLoader
# docs = loader.load()

docs = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

print(f"Original: {len(docs)} docs")

# === 1. CHARACTER SPLITTER (Simple, fixed-size) ===
splitter1 = CharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20,     # Overlap for context
    separator="\n\n",     # Try paragraphs first
    length_function=len,
)
chunks1 = splitter1.split_documents(docs)

# === 2. RECURSIVE (Recommended - respects structure) ===
splitter2 = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""],  # Paragraph > sentence > word
)
chunks2 = splitter2.split_documents(docs)

# === 3. SEMANTIC (Meaning-aware splits) ===
embeddings = OpenAIEmbeddings()
splitter3 = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="standard_deviation",
    breakpoint_threshold_amount=0.5  # Adjust sensitivity
)
chunks3 = splitter3.split_documents(docs)

# Results
print(f"Character chunks: {len(chunks1)}")
print(f"Recursive chunks: {len(chunks2)}")
print(f"Semantic chunks: {len(chunks3)}")
print("\nFirst recursive chunk:")
print(chunks2[1].page_content)

# Sample text demo (no loader needed)
sample = """
Farmers in India work hard planting rice. The IPL cricket league excites millions. Terrorism threatens peace everywhere.
"""
sample_docs = [Document(page_content=sample)]
sample_chunks = splitter2.split_documents(sample_docs)

# Usage notes:
# - chunk_size: Target chunk length (100-1000 typical)
# - chunk_overlap: Context retention (10-20% of size)
# - Recursive > Character for most docs
# - SemanticChunker needs embeddings (great for code/MD)

# Next: VectorStore + Retriever!

#### Step 2: Embedding, Vector store

#### Vector Store Retriever

In [ ]:


from langchain_community.vectorstores import Chroma, FAISS
from langchain_openai import  OpenAIEmbeddings

# Sample documents
documents = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]


embedding_model = OpenAIEmbeddings()


# Step 3: Create Chroma vector store in memory
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_name="my_collection"
)

# vectorstore.similarity_search(query='Who among these are a bowler?',k=2)

# Step 4: Convert vectorstore into a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [ ]:
query = "What is Chroma used for?"
results = retriever.invoke(query)
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

#### Chroma Vector store

In [ ]:
# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

vector_store = Chroma(
    embedding_function=OpenAIEmbeddings(),
    persist_directory='my_chroma_db',
    collection_name='sample'
)

# add documents
vector_store.add_documents(docs)

# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

# search documents
vector_store.similarity_search(query='Who among these are a bowler?',k=2)

# search with similarity score
vector_store.similarity_search_with_score(query='Who among these are a bowler?',k=2)

# meta-data filtering
vector_store.similarity_search_with_score(query="",filter={"team": "Chennai Super Kings"})

# update documents
updated_doc1 = Document(
    page_content="""Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. 
    He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, 
    Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.""",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='09a39dc6-3ba6-4ea7-927e-fdda591da5e4', document=updated_doc1)

# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

# delete document
vector_store.delete(ids=['09a39dc6-3ba6-4ea7-927e-fdda591da5e4'])

# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

#### MMR

In [ ]:
# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

from langchain_community.vectorstores import FAISS

# Initialize OpenAI embeddings
embedding_model = OpenAIEmbeddings()

# Step 2: Create the FAISS vector store from documents
vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embedding_model
)

In [ ]:
# Enable MMR in the retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",                   # <-- This enables MMR
    search_kwargs={"k": 3, "lambda_mult": 0.5}  # k = top results, lambda_mult = relevance-diversity balance
)

In [ ]:
query = "What is langchain?"
results = retriever.invoke(query)

In [ ]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

#### Multiquery Retriever

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
from langchain.retrievers.multi_query import MultiQueryRetriever

In [ ]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [ ]:
# Initialize OpenAI embeddings
embedding_model = OpenAIEmbeddings()

# Create FAISS vector store
vectorstore = FAISS.from_documents(documents=all_docs, embedding=embedding_model)

In [ ]:
# Create retrievers
similarity_retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [ ]:
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 5}),
    llm=ChatOpenAI(model="gpt-3.5-turbo")
)

In [ ]:
# Query
query = "How to improve energy levels and maintain balance?"

# Retrieve results
similarity_results = similarity_retriever.invoke(query)
multiquery_results= multiquery_retriever.invoke(query)

In [ ]:
for i, doc in enumerate(similarity_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

print("*"*150)

for i, doc in enumerate(multiquery_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

#### ContextualCompressionRetriever

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain_core.documents import Document

In [ ]:
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [ ]:
# Create a FAISS vector store from the documents
embedding_model = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(docs, embedding_model)

base_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
compressor = LLMChainExtractor.from_llm(model)

In [ ]:
# Create the contextual compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

In [ ]:
# Query the retriever
query = "What is photosynthesis?"
compressed_results = compression_retriever.invoke(query)

In [ ]:
for i, doc in enumerate(compressed_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


### RAG

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

In [ ]:
video_id = "Gfr50f6ZBvo"  # only the ID, not full URL

try:
    # Fetch transcript in English
    ytt_api = YouTubeTranscriptApi()
    transcript_list = ytt_api.fetch(video_id= "Gfr50f6ZBvo", languages=["en"])

    # Flatten it to plain text
    transcript = " ".join([chunk.text for chunk in transcript_list])
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

except NoTranscriptFound:
    print("No transcript found in the requested language(s).")


In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])
len(chunks)

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = FAISS.from_documents(chunks, embeddings)
# vector_store.index_to_docstore_id
# vector_store.get_by_ids(['2436bdb8-3f5f-49c6-8915-0c654c888700'])

In [ ]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})
retriever.invoke('What is deepmind')

In [ ]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [ ]:
question          = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [ ]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
final_prompt = prompt.invoke({"context": context_text, "question": question})
answer = model.invoke(final_prompt)
print(answer.content)

In [ ]:
# Chain 

from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})
parallel_chain.invoke('who is Demis')

In [ ]:
parser = StrOutputParser()
main_chain = parallel_chain | prompt | model | parser
main_chain.invoke('Can you summarize the video')

### -- END --